# Spatial Residual-Correlation Screen

Fifth alignment-track experiment. Before trying regression kriging or spatial GP residuals, this notebook tests whether well-level residual bias from the current Huber + additive Hilbert model is spatially correlated under fold-held-out validation.

## 1. Setup

In [1]:
"Configure imports, paths, and spatial residual-correction candidates."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = "2026-06-23"
RUN_ID = f"{DATE_STAMP}__spatial_residual_correlation"
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "train"
ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
LOCAL_REFERENCE_RMSE = 14.3586
PUBLIC_REFERENCE_RMSE = 14.514
OFFSET_CLIP = 8.0

CANDIDATES = [{"name": "baseline_no_spatial", "kind": "baseline", "k": 0, "shrink": 0.0}]
for k in [1, 3, 5, 8, 12]:
    for shrink in [0.05, 0.10, 0.20, 0.35]:
        CANDIDATES.append({"name": f"spatial_knn_k{k}_s{int(shrink*1000):03d}", "kind": "knn", "k": k, "shrink": shrink})

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")
print(f"candidate_count={len(CANDIDATES)}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-23__spatial_residual_correlation
candidate_count=21


## 2. Load Residuals And Well Coordinates

In [2]:
"Load residuals and build test-safe spatial/trajectory descriptors per well."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()
for col in ["fold", "row_idx", "eval_progress", "distance_from_boundary_md", "abs_projected_delta_10", "y_true", "y_pred", "residual"]:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")
residuals["fold"] = residuals["fold"].astype(int)
residuals["row_idx"] = residuals["row_idx"].astype(int)
residuals["baseline_pred"] = residuals["y_pred"]
residuals["baseline_residual"] = residuals["y_true"] - residuals["baseline_pred"]


def build_well_descriptor(wellname):
    horizontal = pd.read_csv(DATA_DIR / f"{wellname}__horizontal_well.csv")
    known = horizontal.loc[horizontal["TVT_input"].notna()].copy()
    if len(known) == 0:
        known = horizontal.head(1).copy()
    first = horizontal.iloc[0]
    known_last = known.iloc[-1]
    descriptor = {
        "wellname": wellname,
        "surface_x": float(first["X"]),
        "surface_y": float(first["Y"]),
        "surface_z": float(first["Z"]),
        "last_known_x": float(known_last["X"]),
        "last_known_y": float(known_last["Y"]),
        "last_known_z": float(known_last["Z"]),
        "last_known_tvt": float(known_last["TVT_input"]) if pd.notna(known_last["TVT_input"]) else np.nan,
        "known_md_span": float(known["MD"].max() - known["MD"].min()) if len(known) else np.nan,
        "known_tvt_span": float(known["TVT_input"].max() - known["TVT_input"].min()) if known["TVT_input"].notna().any() else np.nan,
        "xy_span_known": float(np.hypot(known["X"].max() - known["X"].min(), known["Y"].max() - known["Y"].min())) if len(known) else np.nan,
    }
    return descriptor

well_descriptors = pd.DataFrame([build_well_descriptor(well) for well in sorted(residuals["wellname"].unique())])
well_residuals = (
    residuals.groupby(["fold", "wellname"], as_index=False)
    .agg(
        mean_residual=("baseline_residual", "mean"),
        mae=("baseline_residual", lambda s: float(np.mean(np.abs(s)))),
        rmse=("baseline_residual", lambda s: float(np.sqrt(np.mean(np.asarray(s) ** 2)))),
        n_rows=("baseline_residual", "size"),
    )
    .merge(well_descriptors, on="wellname", how="left")
)
print(f"rows={len(residuals):,}")
print(f"wells={well_residuals['wellname'].nunique():,}")
print(well_residuals[["mean_residual", "mae", "rmse"]].describe().round(3).to_string())

rows=5,400
wells=773
       mean_residual      mae     rmse
count        773.000  773.000  773.000
mean           0.258   10.416   12.238
std           11.894    7.385    8.234
min          -46.213    1.172    1.543
25%           -6.791    5.656    6.901
50%            0.122    8.356   10.013
75%            6.989   13.256   15.476
max           56.454   56.454   60.918


## 3. Fold-Held-Out Spatial Diagnostics

In [3]:
"Measure nearest-neighbor residual association and semivariance by distance bin."

SPATIAL_FEATURES = ["surface_x", "surface_y", "last_known_x", "last_known_y", "last_known_tvt", "known_md_span", "known_tvt_span", "xy_span_known"]

def scaled_features(train_wells, query_wells):
    med = train_wells[SPATIAL_FEATURES].median(numeric_only=True)
    train_x = train_wells[SPATIAL_FEATURES].fillna(med).to_numpy(dtype=float)
    query_x = query_wells[SPATIAL_FEATURES].fillna(med).to_numpy(dtype=float)
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_x)
    query_scaled = scaler.transform(query_x)
    return train_scaled, query_scaled

neighbor_rows = []
variogram_rows = []
for fold in sorted(well_residuals["fold"].unique()):
    train_wells = well_residuals.loc[~well_residuals["fold"].eq(fold)].copy().reset_index(drop=True)
    valid_wells = well_residuals.loc[well_residuals["fold"].eq(fold)].copy().reset_index(drop=True)
    train_x, valid_x = scaled_features(train_wells, valid_wells)
    distances = np.sqrt(((valid_x[:, None, :] - train_x[None, :, :]) ** 2).mean(axis=2))
    nearest_idx = np.argmin(distances, axis=1)
    for i, row in valid_wells.iterrows():
        nearest = train_wells.iloc[int(nearest_idx[i])]
        neighbor_rows.append(
            {
                "fold": int(fold),
                "wellname": row["wellname"],
                "nearest_well": nearest["wellname"],
                "descriptor_distance": float(distances[i, nearest_idx[i]]),
                "valid_mean_residual": float(row["mean_residual"]),
                "nearest_mean_residual": float(nearest["mean_residual"]),
                "residual_diff": float(row["mean_residual"] - nearest["mean_residual"]),
            }
        )
        order = np.argsort(distances[i])[:12]
        for rank, pos in enumerate(order, start=1):
            train_row = train_wells.iloc[int(pos)]
            variogram_rows.append(
                {
                    "fold": int(fold),
                    "wellname": row["wellname"],
                    "neighbor_rank": rank,
                    "descriptor_distance": float(distances[i, pos]),
                    "semivariance": float(0.5 * (row["mean_residual"] - train_row["mean_residual"]) ** 2),
                }
            )

neighbor_diagnostics = pd.DataFrame(neighbor_rows)
variogram = pd.DataFrame(variogram_rows)
neighbor_corr = neighbor_diagnostics[["valid_mean_residual", "nearest_mean_residual"]].corr().iloc[0, 1]
variogram["distance_bin"] = pd.qcut(variogram["descriptor_distance"], q=6, duplicates="drop")
variogram_summary = variogram.groupby("distance_bin", observed=False).agg(mean_distance=("descriptor_distance", "mean"), mean_semivariance=("semivariance", "mean"), rows=("semivariance", "size")).reset_index()
print(f"nearest_residual_corr={neighbor_corr:.4f}")
print(variogram_summary.to_string(index=False))

nearest_residual_corr=0.0422
      distance_bin  mean_distance  mean_semivariance  rows
(-0.000804, 0.221]       0.164155         120.343868  1546
    (0.221, 0.275]       0.250607         114.655287  1546
    (0.275, 0.325]       0.299105         111.207364  1546
    (0.325, 0.381]       0.352002         140.050494  1546
     (0.381, 0.48]       0.424696         145.315137  1546
     (0.48, 1.506]       0.635092         175.465796  1546


## 4. Spatial KNN Residual Correction

In [4]:
"Test conservative spatial KNN well-bias corrections on held-out folds."

start_time = time.time()
metric_rows = []
prediction_rows = []
for candidate in CANDIDATES:
    for fold in sorted(residuals["fold"].unique()):
        train_wells = well_residuals.loc[~well_residuals["fold"].eq(fold)].copy().reset_index(drop=True)
        valid_wells = well_residuals.loc[well_residuals["fold"].eq(fold)].copy().reset_index(drop=True)
        valid_rows = residuals.loc[residuals["fold"].eq(fold)].copy()
        corrections_by_well = {well: 0.0 for well in valid_wells["wellname"]}
        if candidate["kind"] == "knn":
            train_x, valid_x = scaled_features(train_wells, valid_wells)
            distances = np.sqrt(((valid_x[:, None, :] - train_x[None, :, :]) ** 2).mean(axis=2))
            for i, row in valid_wells.iterrows():
                order = np.argsort(distances[i])[:candidate["k"]]
                picked = train_wells.iloc[order]
                d = distances[i, order]
                weights = 1.0 / np.maximum(d, 1e-6)
                weights = weights / weights.sum()
                correction = float(np.sum(picked["mean_residual"].to_numpy(dtype=float) * weights) * candidate["shrink"])
                corrections_by_well[row["wellname"]] = float(np.clip(correction, -OFFSET_CLIP, OFFSET_CLIP))
        correction = valid_rows["wellname"].map(corrections_by_well).fillna(0.0).to_numpy(dtype=float)
        y_true = valid_rows["y_true"].to_numpy()
        y_pred = valid_rows["baseline_pred"].to_numpy() + correction
        metric_rows.append(
            {
                "candidate": candidate["name"],
                "kind": candidate["kind"],
                "fold": int(fold),
                "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                "mae": float(mean_absolute_error(y_true, y_pred)),
                "n_rows": int(len(valid_rows)),
                "n_wells": int(valid_rows["wellname"].nunique()),
                "mean_correction": float(np.mean(correction)),
                "std_correction": float(np.std(correction)),
                "max_abs_correction": float(np.max(np.abs(correction))) if len(correction) else 0.0,
            }
        )
        out = valid_rows[["fold", "wellname", "row_idx", "y_true", "baseline_pred", "baseline_residual", "eval_progress"]].copy()
        out["candidate"] = candidate["name"]
        out["spatial_correction"] = correction
        out["y_pred"] = y_pred
        out["residual_after_spatial"] = out["y_true"] - out["y_pred"]
        prediction_rows.append(out)

fold_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
summary = (
    fold_metrics.groupby(["candidate", "kind"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        mean_correction=("mean_correction", "mean"),
        std_correction=("std_correction", "mean"),
        max_abs_correction=("max_abs_correction", "max"),
        n_rows=("n_rows", "sum"),
        n_wells=("n_wells", "sum"),
    )
)
baseline_rmse = float(summary.loc[summary["candidate"].eq("baseline_no_spatial"), "rmse_mean"].iloc[0])
summary["rmse_delta_vs_sample_baseline"] = summary["rmse_mean"] - baseline_rmse
summary = summary.sort_values(["rmse_mean", "candidate"]).reset_index(drop=True)
train_time_s = time.time() - start_time
print(summary.head(15).to_string(index=False))

           candidate     kind  rmse_mean  rmse_std  mae_mean  mean_correction  std_correction  max_abs_correction  n_rows  n_wells  rmse_delta_vs_sample_baseline
spatial_knn_k12_s200      knn  14.714428  0.432910 10.404505         0.028897        0.795341            3.206953    5400      773                      -0.024117
spatial_knn_k12_s100      knn  14.721092  0.459921 10.401811         0.014448        0.397670            1.603476    5400      773                      -0.017453
spatial_knn_k12_s350      knn  14.724675  0.393333 10.432449         0.050569        1.391846            5.612167    5400      773                      -0.013870
 spatial_knn_k8_s100      knn  14.728143  0.458547 10.411315         0.013753        0.470108            1.606819    5400      773                      -0.010402
spatial_knn_k12_s050      knn  14.728471  0.473489 10.405944         0.007224        0.198835            0.801738    5400      773                      -0.010074
 spatial_knn_k1_s050      kn

## 5. Diagnostics And Export

In [5]:
"Export spatial diagnostics, correction results, and run bundle."

best_candidate = summary.iloc[0]["candidate"]
best_row = summary.iloc[0]
baseline_row = summary.loc[summary["candidate"].eq("baseline_no_spatial")].iloc[0]
best_predictions = predictions.loc[predictions["candidate"].eq(best_candidate)].copy()
well_before = residuals.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_before=("baseline_residual", "mean"))
well_after = best_predictions.groupby(["fold", "wellname"], as_index=False).agg(mean_residual_after=("residual_after_spatial", "mean"), mean_spatial_correction=("spatial_correction", "mean"))
well_diagnostics = well_after.merge(well_before, on=["fold", "wellname"], how="left")
well_diagnostics["abs_bias_delta"] = well_diagnostics["mean_residual_after"].abs() - well_diagnostics["mean_residual_before"].abs()

findings = pd.DataFrame(
    [
        {"finding": "nearest_residual_corr", "value": f"{neighbor_corr:.4f}", "modeling_implication": "Positive correlation supports spatial/descriptor residual pooling; near zero argues against kriging."},
        {"finding": "best_candidate", "value": f"{best_candidate} RMSE {best_row['rmse_mean']:.4f} vs sample baseline {baseline_row['rmse_mean']:.4f}", "modeling_implication": "Promote spatial residual correction only if grouped RMSE improves clearly and folds are stable."},
        {"finding": "sample_baseline_delta", "value": f"best candidate delta {best_row['rmse_delta_vs_sample_baseline']:.4f}", "modeling_implication": "Small or positive deltas mean spatial residual structure is diagnostic-only for now."},
        {"finding": "well_bias_change", "value": f"median absolute well-bias delta {well_diagnostics['abs_bias_delta'].median():.4f}", "modeling_implication": "Negative is good; positive means spatial corrections add well-level noise."},
    ]
)

summary_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_summary__{DATE_STAMP}.csv"
fold_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_fold_metrics__{DATE_STAMP}.csv"
pred_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_residual_sample__{DATE_STAMP}.csv"
well_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_well_diagnostics__{DATE_STAMP}.csv"
neighbor_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_neighbors__{DATE_STAMP}.csv"
variogram_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_variogram__{DATE_STAMP}.csv"
variogram_summary_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_variogram_summary__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__spatial_residual_correlation_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__spatial_residual_correlation__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / "model_comparison.csv"

summary.to_csv(summary_path, index=False)
fold_metrics.to_csv(fold_path, index=False)
predictions.to_csv(pred_path, index=False)
well_diagnostics.to_csv(well_path, index=False)
neighbor_diagnostics.to_csv(neighbor_path, index=False)
variogram.to_csv(variogram_path, index=False)
variogram_summary.to_csv(variogram_summary_path, index=False)
findings.to_csv(findings_path, index=False)

report = f"""# Spatial Residual-Correlation Screen

Run ID: `{RUN_ID}`

Result

Best candidate: `{best_candidate}`

- Nearest-neighbor residual correlation: **{neighbor_corr:.4f}**
- Same-sample baseline RMSE: **{baseline_row['rmse_mean']:.4f}**
- Best RMSE: **{best_row['rmse_mean']:.4f}**
- Delta vs same-sample baseline: **{best_row['rmse_delta_vs_sample_baseline']:.4f}**
- Current full local reference: **{LOCAL_REFERENCE_RMSE:.4f}**
- Current public reference: **{PUBLIC_REFERENCE_RMSE:.3f}**

Interpretation

This screen checks whether residuals justify regression kriging or spatial GP work. It is a residual-sample diagnostic, not a Kaggle submission notebook.
"""
report_path.write_text(report, encoding="utf-8")

comparison_row = pd.DataFrame(
    [
        {
            "run_id": RUN_ID,
            "model_family": "alignment_spatial_residual",
            "model_name": best_candidate,
            "cv_rmse_mean": float(best_row["rmse_mean"]),
            "cv_rmse_std": float(best_row["rmse_std"]),
            "train_time_s": float(train_time_s),
            "n_features": int(len(SPATIAL_FEATURES)),
            "notes": "Fold-safe spatial/trajectory descriptor residual-correlation and KNN well-bias correction screen",
        }
    ]
)
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
if len(comparison):
    comparison = comparison.loc[~comparison["run_id"].eq(RUN_ID)]
comparison = pd.concat([comparison, comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

for src in [summary_path, fold_path, pred_path, well_path, neighbor_path, variogram_path, variogram_summary_path, findings_path, report_path, comparison_path]:
    shutil.copy2(src, ARTIFACT_LOGS / src.name)

config = {"run_id": RUN_ID, "target_candidate": TARGET_CANDIDATE, "offset_clip": OFFSET_CLIP, "candidates": CANDIDATES, "spatial_features": SPATIAL_FEATURES, "local_reference_rmse": LOCAL_REFERENCE_RMSE, "public_reference_rmse": PUBLIC_REFERENCE_RMSE}
(ARTIFACT_LOGS / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

print(findings.to_string(index=False))
print(f"wrote {summary_path}")
print(f"wrote run bundle {ARTIFACT_LOGS}")

              finding                                                        value                                                                                 modeling_implication
nearest_residual_corr                                                       0.0422 Positive correlation supports spatial/descriptor residual pooling; near zero argues against kriging.
       best_candidate spatial_knn_k12_s200 RMSE 14.7144 vs sample baseline 14.7385      Promote spatial residual correction only if grouped RMSE improves clearly and folds are stable.
sample_baseline_delta                                 best candidate delta -0.0241                 Small or positive deltas mean spatial residual structure is diagnostic-only for now.
     well_bias_change                      median absolute well-bias delta -0.0073                           Negative is good; positive means spatial corrections add well-level noise.
wrote C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\art

## 6. Conclusion

In [6]:
"Print final decision summary."

print(f"nearest_residual_corr={neighbor_corr:.4f}")
print(f"best_candidate={best_candidate}")
print(f"sample_baseline_rmse={baseline_row['rmse_mean']:.4f}")
print(f"best_rmse={best_row['rmse_mean']:.4f}")
print(f"delta_vs_sample_baseline={best_row['rmse_delta_vs_sample_baseline']:.4f}")
print(f"promotion_gate_cleared={best_row['rmse_delta_vs_sample_baseline'] <= -0.05}")

nearest_residual_corr=0.0422
best_candidate=spatial_knn_k12_s200
sample_baseline_rmse=14.7385
best_rmse=14.7144
delta_vs_sample_baseline=-0.0241
promotion_gate_cleared=False
